# NALAPRO Project — Part 4
## Llama-3 — Zero-Shot and Few-Shot Classification

---

## Tools used
| Tool | Purpose |
|------|---------|
| `transformers` (alternative) | HuggingFace inference for Llama-3 |
| `scikit-learn` | Evaluation metrics, confusion matrix |
| `matplotlib` | Visualisations |
| `pandas` | Results tables |
| `Claude (Anthropic)` | Scaffolding and code assistance |

**Reference:** *Hands-On Large Language Models* — https://github.com/HandsOnLLM/Hands-On-Large-Language-Models  
**In-class notebook:** `AdvancedTechniques_LangChain.ipynb` (Phi-3 via LangChain)  
**Prompt engineering:** Ch. 6, *Hands-On LLMs*

---

### What is an LLM doing when it classifies?

Llama-3 is a *decoder-only* language model. It was not trained on 20-Newsgroups —
it has never seen our category labels, and its parameters are *never updated* in this notebook.
Yet it can classify newsgroup posts because:

### Zero-shot vs. few-shot

| Regime | Prompt contains | Labelled data needed |
|--------|----------------|---------------------|
| **Zero-shot** | Task description + class list | 0 examples |
| **Few-shot** | Task description + class list + k examples per class | k × 20 examples |

---

## Setup


In [ ]:
%pip install -q requests scikit-learn matplotlib pandas tqdm


---
## Experiment tracking (in-notebook)

Part 4 has no gradient-based training, so there are no weights or parameter tables.
Instead the **inference results** are captured directly in this notebook:

In [ ]:
import re, os, random, json
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# ── Google Drive setup (Google Colab only) ────────────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    NALAPRO_DIR = '/content/drive/MyDrive/NALAPRO'
    os.makedirs(NALAPRO_DIR, exist_ok=True)
    os.chdir(NALAPRO_DIR)
    _IN_COLAB = True
    print(f'Google Colab detected.')
    print(f'Working directory → {NALAPRO_DIR}')
    print('Expected files from prior parts: part1_results.csv, part2_results.csv, part3_results.csv')
    missing = [f for f in ['part1_results.csv','part2_results.csv','part3_results.csv']
               if not os.path.exists(f)]
    if missing:
        print(f'WARNING: missing prior-part CSVs: {missing}')
        print('  → Run Parts 1–3 first (in the same Drive folder) or placeholder values will be used.')
except ModuleNotFoundError:
    _IN_COLAB = False
    print(f'Not in Colab. Working directory: {os.getcwd()}')

---
## 1 · Data


In [ ]:
from sklearn.datasets import fetch_20newsgroups

REMOVE = ("headers", "footers", "quotes")
ng_train = fetch_20newsgroups(subset="train", remove=REMOVE, random_state=SEED)
ng_test  = fetch_20newsgroups(subset="test",  remove=REMOVE, random_state=SEED)

class_names = ng_train.target_names
NUM_CLASSES = len(class_names)
print("Number of classes:", NUM_CLASSES)
print("Classes:", class_names)


In [ ]:
# ── Build stratified evaluation set ──────────────────────────────────────────────
N_PER_CLASS = 20

by_class = defaultdict(list)
for text, label in zip(ng_test.data, ng_test.target):
    if text and len(text.strip()) > 10:
        by_class[label].append(text)

rng = random.Random(SEED)
eval_texts, eval_labels = [], []
for c in range(NUM_CLASSES):
    pool  = by_class[c]
    picks = rng.sample(pool, min(N_PER_CLASS, len(pool)))
    eval_texts.extend(picks)
    eval_labels.extend([c] * len(picks))

print(f"Evaluation set: {len(eval_texts)} documents  ({N_PER_CLASS} per class)")


In [ ]:
# ── Build few-shot pool from the TRAINING split ──────────────────────────────────
fewshot_pool = defaultdict(list)
for text, label in zip(ng_train.data, ng_train.target):
    # Keep only short documents so the few-shot prompt does not overflow the context window
    if text and 50 < len(text) < 600:
        fewshot_pool[label].append(text.strip())

for c in range(NUM_CLASSES):
    print(f"  {class_names[c]:35s}  {len(fewshot_pool[c]):4d} short examples available")


---
## 2 · Backend — pick ONE option

| **C — Groq API** | Llama-3-8B (remote, free) | **fast, free, no gating** |

> **uses Groq** — a free inference API that serves the original Llama-3-8B-8192 model.
> One-time setup: sign up at [console.groq.com](https://console.groq.com) (free, no credit card),
> create an API key, then add it in Colab Secrets as `GROQ_API_KEY`.

In [ ]:
# ONE-TIME SETUP:
#   1. Go to https://console.groq.com → sign up
#   2. Click "API Keys" → "Create API key" → copy the key (starts with gsk_)
#   3. In Colab: left sidebar → 🔑 Secrets → New secret
#      Name: GROQ_API_KEY   Value: gsk_...
# ─────────────────────────────────────────────────────────────────────────────────
import os, re, time, requests

# ── Read API key ──────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY', '')

if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found.\n"
        "  1. Sign up free at https://console.groq.com\n"
        "  2. API Keys → Create API key → copy it\n"
        "  3. Colab: left sidebar → 🔑 Secrets → add 'GROQ_API_KEY'"
    )

# ── Auto-select the best available model ─────────────────────────────────────────
PREFERRED = [
    "llama-3.1-8b-instant",
    "llama3-8b-8192",
    "llama-3.3-70b-versatile",
    "llama3-70b-8192",
    "llama-3.1-70b-versatile",
    "mixtral-8x7b-32768",
    "gemma2-9b-it",
]

_models_resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
    timeout=10,
)
if not _models_resp.ok:
    raise RuntimeError(f"Could not list Groq models: {_models_resp.status_code} {_models_resp.text}")

_available = {m["id"] for m in _models_resp.json()["data"]}
print("Available Groq models:", sorted(_available))

GROQ_MODEL = next((m for m in PREFERRED if m in _available), sorted(_available)[0])
print(f"\nUsing model: {GROQ_MODEL}")

# ── Generate function with automatic rate-limit back-off ─────────────────────────
def generate(prompt: str, max_new_tokens: int = 25) -> str:
    """
    Call Groq's API. On 429 rate-limit, reads the 'try again in Xs' wait time
    from the error message and sleeps exactly that long before retrying.
    """
    for attempt in range(15):
        resp = requests.post(
            "https://api.groq.com/openai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {GROQ_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": GROQ_MODEL,
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_new_tokens,
                "temperature": 0.001,
            },
            timeout=30,
        )
        if resp.status_code == 429:
            body = resp.json()
            msg  = body.get("error", {}).get("message", "")
            m    = re.search(r"try again in (\d+(?:\.\d+)?)\s*s", msg)
            wait = float(m.group(1)) + 0.5 if m else 5.0
            time.sleep(wait)
            continue
        if not resp.ok:
            raise RuntimeError(f"Groq API error {resp.status_code}: {resp.text}")
        return resp.json()["choices"][0]["message"]["content"].strip()
    raise RuntimeError("Groq rate limit: max retries exceeded — try again in a minute.")

# Smoke test
print("Backend test:", generate("Reply with exactly one word: pong"))

---
## 3 · Prompt engineering

### Why prompt design matters

The model does not "know" what format we want the output in.
Without explicit constraints, it might output:
- `"This post belongs to the comp.graphics category."` — verbose, parseable
- `"comp.graphics"` — exactly what we want
- `"Computer Graphics"` — capitalized differently, might not match
- `"I'm not sure, but it could be sci.med or sci.crypt"` — ambiguous

By explicitly saying *"Output ONLY the category name on a single line"*, we dramatically
reduce the variance in output format.

**Reference:** *Hands-On Large Language Models*, Ch. 6 — Prompt Engineering  
→ https://github.com/HandsOnLLM/Hands-On-Large-Language-Models


In [ ]:
# Build the class list once — reused in both prompts
CLASS_LIST = "\n".join(f"  {c}" for c in class_names)

ZERO_SHOT_TEMPLATE = (
    "You are a precise text classifier.\n"
    "Classify the following Usenet newsgroup post into exactly ONE of the 20 categories listed below.\n"
    "Output ONLY the category name on a single line — nothing else, no explanation, no punctuation.\n"
    "\nCategories:\n{class_list}\n"
    "\nPost to classify:\n\"\"\"\n{post}\n\"\"\"\n"
    "\nCategory:"
)

FEW_SHOT_HEADER = (
    "You are a precise text classifier.\n"
    "Classify each Usenet newsgroup post into exactly ONE of the 20 categories listed below.\n"
    "Output ONLY the category name on a single line — nothing else.\n"
    "\nCategories:\n{class_list}\n"
    "\nHere are some labelled examples to help you understand the format and content of each category:\n"
)

def build_zero_shot_prompt(post: str) -> str:
    """Construct a zero-shot classification prompt. Truncate long posts to 2000 chars."""
    return ZERO_SHOT_TEMPLATE.format(class_list=CLASS_LIST, post=post[:2000])

def build_few_shot_prompt(post: str, k: int = 1) -> str:
    """
    Construct a few-shot classification prompt with k examples per class.
    k=1 → 20 total examples (one per class)
    k=2 → 40 total examples
    Note: with k=2 and long documents, prompts can exceed 4000 tokens.
    """
    parts = [FEW_SHOT_HEADER.format(class_list=CLASS_LIST), "\n"]
    for c in range(NUM_CLASSES):
        for ex in fewshot_pool[c][:k]:
            parts.append(f"Post:\n\"\"\"\n{ex[:400]}\n\"\"\"\n"
                         f"Category: {class_names[c]}\n\n")
    parts.append(f"Now classify this new post:\n"
                 f"Post:\n\"\"\"\n{post[:1500]}\n\"\"\"\n"
                 f"\nCategory:")
    return "".join(parts)

# Show prompt lengths as a sanity check
sample = eval_texts[0]
zs_prompt = build_zero_shot_prompt(sample)
fs_prompt = build_few_shot_prompt(sample, k=1)
print(f"Zero-shot prompt length : {len(zs_prompt):,} chars")
print(f"Few-shot  prompt length : {len(fs_prompt):,} chars  (k=1 per class = 20 examples total)")


In [ ]:
def parse_label(raw: str) -> int:
    """
    Robust label parser. Tries three strategies and returns the class index (0–19)
    or -1 if no match is found.

    Why is robustness important?
    Even with a well-crafted prompt, instruction-tuned models sometimes add extra words,
    change capitalisation, or output a paraphrase. We handle the most common deviations
    without being so lenient that we accept wrong answers.
    """
    if not raw:
        return -1
    # Take only the first line — ignore any explanation the model added after
    text = raw.strip().splitlines()[0].strip().rstrip(".,:;!?")

    # 1. Exact match (case-insensitive)
    for i, c in enumerate(class_names):
        if text.lower() == c.lower():
            return i

    # 2. Substring match — the class name is somewhere in the response
    text_lower = text.lower()
    for i, c in enumerate(class_names):
        if c.lower() in text_lower:
            return i

    # 3. Token-level match — split on whitespace and punctuation, try each token
    tokens = re.split(r"[\s/,;:]+", text_lower)
    for tok in tokens:
        for i, c in enumerate(class_names):
            if tok == c.lower():
                return i

    return -1   # unparseable

# Validate the parser on synthetic cases
test_cases = [
    ("rec.sport.hockey", 10),
    ("Category: sci.space", 14),
    ("I believe it's comp.graphics.", 1),
    ("This is a tough one. Maybe alt.atheism?", 0),
    ("I don't know", -1),
    ("soc.religion.christian is my answer", 15),
]
print("Parser validation:")
for raw, expected_idx in test_cases:
    parsed = parse_label(raw)
    expected_name = class_names[expected_idx] if expected_idx >= 0 else "UNPARSEABLE"
    parsed_name   = class_names[parsed]  if parsed >= 0  else "UNPARSEABLE"
    status = "✓" if parsed == expected_idx else "✗"
    print(f"  {status}  Input: {repr(raw):50s}  Expected: {expected_name:30s}  Got: {parsed_name}")


---
## 4 · Zero-shot evaluation

In [ ]:
zs_preds, zs_raw_outputs = [], []

for post in tqdm(eval_texts, desc="Zero-shot inference"):
    prompt = build_zero_shot_prompt(post)
    try:
        raw = generate(prompt, max_new_tokens=25)
    except Exception as e:
        print("Inference error:", e)
        raw = ""
    zs_raw_outputs.append(raw)
    zs_preds.append(parse_label(raw))

zs_preds = np.array(zs_preds)
y_true   = np.array(eval_labels)

unparseable = (zs_preds == -1).sum()
print(f"\nUnparseable responses : {unparseable} / {len(zs_preds)}  ({100*unparseable/len(zs_preds):.1f}%)")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Replace unparseable predictions with a dummy class (NUM_CLASSES) — always wrong
preds_safe_zs = np.where(zs_preds == -1, NUM_CLASSES, zs_preds)

zs_acc            = accuracy_score(y_true, preds_safe_zs)
zs_f1             = f1_score(y_true, preds_safe_zs, average="macro",
                              labels=list(range(NUM_CLASSES)))
zs_unparseable_pct = 100 * (zs_preds == -1).sum() / len(zs_preds)

print(f"Zero-shot → accuracy = {zs_acc:.4f}   macro-F1 = {zs_f1:.4f}   unparseable = {zs_unparseable_pct:.1f}%")

---
## 5 · Few-shot evaluation (k=1 example per class = 20 examples total)


In [ ]:
K = 1   # examples per class — you can experiment with K=2 or K=3

fs_preds, fs_raw_outputs = [], []

for post in tqdm(eval_texts, desc=f"Few-shot (k={K}) inference"):
    prompt = build_few_shot_prompt(post, k=K)
    try:
        raw = generate(prompt, max_new_tokens=25)
    except Exception as e:
        print("Inference error:", e)
        raw = ""
    fs_raw_outputs.append(raw)
    fs_preds.append(parse_label(raw))

fs_preds = np.array(fs_preds)
unparseable_fs = (fs_preds == -1).sum()
print(f"\nUnparseable responses : {unparseable_fs} / {len(fs_preds)}  ({100*unparseable_fs/len(fs_preds):.1f}%)")


In [ ]:
preds_safe_fs = np.where(fs_preds == -1, NUM_CLASSES, fs_preds)

fs_acc            = accuracy_score(y_true, preds_safe_fs)
fs_f1             = f1_score(y_true, preds_safe_fs, average="macro",
                              labels=list(range(NUM_CLASSES)))
fs_unparseable_pct = 100 * (fs_preds == -1).sum() / len(fs_preds)

print(f"Few-shot (k={K}) → accuracy = {fs_acc:.4f}   macro-F1 = {fs_f1:.4f}   unparseable = {fs_unparseable_pct:.1f}%")

---
## 6 · Comprehensive analysis & comparison


In [ ]:
import pandas as pd, matplotlib.pyplot as plt

# ── Helpers to read results from prior parts ──────────────────────────────────────
def _best_row(path, f1_col="Macro-F1"):
    """
    Load a CSV and return the row with the highest F1.
    Values may be stored as formatted strings (e.g. '0.8543') — pd.to_numeric handles that.
    Falls back to the last column if f1_col is not present.
    """
    try:
        df = pd.read_csv(path)
        col = f1_col if f1_col in df.columns else df.columns[-1]
        return df.loc[pd.to_numeric(df[col], errors="coerce").idxmax()]
    except Exception:
        return None

def _get(row, *keys):
    """Return the first matching key from a pandas Series, converted to float if possible."""
    if row is None:
        return None
    for k in keys:
        if k in row.index:
            try:
                return float(row[k])
            except (ValueError, TypeError):
                return row[k]
    return None

# ── Load best rows from Parts 1, 2, 3 ────────────────────────────────────────────
p1 = _best_row("part1_results.csv", f1_col="Macro-F1")
p2 = _best_row("part2_results.csv", f1_col="Test Macro-F1")
p3 = _best_row("part3_results.csv", f1_col="Test F1")

RESULTS = [
    {
        "Experiment"     : f"Part 1 — {_get(p1, 'Experiment') or 'best'} + SimpleNN",
        "Test Accuracy"  : _get(p1, "Test Acc", "Test Accuracy") or 0.0,
        "Test Macro-F1"  : _get(p1, "Macro-F1", "Test Macro-F1") or 0.0,
        "Unparseable %"  : None,
        "Trainable Params": "N/A",
    },
    {
        "Experiment"     : f"Part 2 — {_get(p2, 'Experiment') or 'best run'}",
        "Test Accuracy"  : _get(p2, "Test Acc", "Test Accuracy") or 0.0,
        "Test Macro-F1"  : _get(p2, "Test Macro-F1", "Test F1") or 0.0,
        "Unparseable %"  : None,
        "Trainable Params": "N/A",
    },
    {
        "Experiment"     : "Part 3 — MLM-adapted BERT",
        "Test Accuracy"  : _get(p3, "Test Acc", "Test Accuracy") or 0.0,
        "Test Macro-F1"  : _get(p3, "Test F1", "Test Macro-F1") or 0.0,
        "Unparseable %"  : None,
        "Trainable Params": "N/A",
    },
    {
        "Experiment"     : "Part 4 — Llama-3 zero-shot",
        "Test Accuracy"  : zs_acc,
        "Test Macro-F1"  : zs_f1,
        "Unparseable %"  : round(zs_unparseable_pct, 1),
        "Trainable Params": 0,
    },
    {
        "Experiment"     : f"Part 4 — Llama-3 {K}-shot/class",
        "Test Accuracy"  : fs_acc,
        "Test Macro-F1"  : fs_f1,
        "Unparseable %"  : round(fs_unparseable_pct, 1),
        "Trainable Params": 0,
    },
]

summary = pd.DataFrame(RESULTS)
print("\n── Cross-part results summary ───────────────────────────────────────────")
print(summary.to_string(index=False))
summary.to_csv("part4_results_all.csv", index=False)
print("\n→ Saved part4_results_all.csv")

In [ ]:
# ── Side-by-side bar chart of all five systems ───────────────────────────────────
labels = [r.replace(" — ", "\n") for r in summary["Experiment"]]
accs   = summary["Test Accuracy"].tolist()
f1s    = summary["Test Macro-F1"].tolist()
colors = ["#FF6B35", "#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

x = range(len(summary))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, vals, title in [(axes[0], accs, "Test Accuracy"), (axes[1], f1s, "Test Macro-F1")]:
    bars = ax.bar(x, vals, color=colors, width=0.55)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8.5, rotation=15, ha="right")
    ax.set_ylim(0, 1.08)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel(title); ax.grid(axis="y", alpha=0.3)

plt.suptitle("Performance across all project parts\n(same 20-Newsgroups test set)", fontsize=13)
plt.tight_layout()
plt.savefig("part4_all_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part4_all_results.png  (use as the master comparison figure in the report)")

In [ ]:
# ── Confusion matrix for the few-shot run ────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

mask = fs_preds != -1   # exclude unparseable
cm = confusion_matrix(y_true[mask], fs_preds[mask],
                      labels=list(range(NUM_CLASSES)), normalize="true")

fig, ax = plt.subplots(figsize=(13, 11))
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(ax=ax, xticks_rotation=90, cmap="Purples", colorbar=False, values_format=".2f")
ax.set_title(f"Llama-3 few-shot (k={K}) — normalised confusion matrix",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("part4_confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("→ Saved part4_confusion_matrix.png")


In [ ]:
# ── Qualitative inspection ─────────────
print("=" * 80)
print("Qualitative examples for the report")
print("=" * 80)
shown = 0
for i in range(len(eval_texts)):
    if shown >= 6:
        break
    zs_c = class_names[zs_preds[i]] if zs_preds[i] >= 0 else "UNPARSED"
    fs_c = class_names[fs_preds[i]] if fs_preds[i] >= 0 else "UNPARSED"
    true_c = class_names[y_true[i]]
    # Show cases where zero-shot and few-shot differ
    if zs_c != fs_c or (i % 60 == 0):
        print(f"\nExample {i}")
        print(f"  True label  : {true_c}")
        print(f"  Zero-shot   : {zs_c}  (raw: {repr(zs_raw_outputs[i][:60])})")
        print(f"  Few-shot    : {fs_c}  (raw: {repr(fs_raw_outputs[i][:60])})")
        print(f"  Post excerpt: {eval_texts[i][:200].replace(chr(10), ' ')}")
        shown += 1


---
## ⚠️ Teardown — Run this before closing the notebook

> ### 🔴 STOP AND READ BEFORE YOU CLOSE
>
> **Google Colab charges per GPU-hour while the runtime is connected.**
> Even if you're not running cells, the runtime keeps billing.
>
> | Time forgotten | Approx cost (Colab Pro, T4 GPU) |
> |---|---|
> | 1 hour | ~$0.35 |
> | 1 day | ~$8 |
> | 1 week | ~$56 |
>
> **Run the two cells below, then disconnect your runtime.**

### What the teardown does
1. **Deletes model objects** from Python memory → frees GPU VRAM immediately
2. **Deletes checkpoint folders** from Colab disk → prevents disk-quota errors on next run
3. **Clears CUDA cache** → returns GPU memory to the OS
4. **Prints a checklist** → step-by-step confirmation you're fully shut down

### After running the teardown cells
Go to **Runtime → Disconnect and delete runtime** in the Colab menu bar.
That stops all billing for this session.



In [ ]:
# ── Step 1: Delete model objects & free GPU VRAM ─────────────────────────────
# If you used the HuggingFace Transformers backend (Option B), the Llama-3
# model is loaded into GPU VRAM and must be explicitly deleted.
# If you used Ollama or the HF Inference API, there is nothing to delete here.
import gc, torch

print('Freeing any loaded LLM from GPU VRAM…')

for _varname in ['hf_model', 'hf_tok']:
    if _varname in globals():
        obj = globals()[_varname]
        if hasattr(obj, 'cpu'):
            obj.cpu()
        del globals()[_varname]
        print(f'  ✓ Deleted {_varname}')

for _varname in ['eval_texts', 'zs_raw_outputs', 'fs_raw_outputs',
                 'zs_preds', 'fs_preds']:
    if _varname in globals():
        del globals()[_varname]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1e6
    print(f'CUDA cache cleared. GPU memory still allocated: {allocated:.1f} MB')
else:
    print('No GPU in use (Ollama / CPU backend) — gc.collect() run.')

print('✓ Step 1 complete.')


In [ ]:
# ── Step 2: Stop Ollama (if running) & check HF Inference API spend ──────────
import subprocess, os

try:
    result = subprocess.run(['pkill', '-f', 'ollama'], capture_output=True, text=True)
    if result.returncode == 0:
        print('✓ Ollama process stopped.')
    else:
        print('  Ollama was not running in this session (OK if you used another backend).')
except FileNotFoundError:
    print('  pkill not available — Ollama was not started in this session.')

print()
print('=' * 65)
print('PART 4 TEARDOWN CHECKLIST')
print('=' * 65)

checks = [
    ('Ollama server stopped (if used)',
     'No billable cloud resource — runs locally'),
    ('HuggingFace Transformers model deleted from GPU VRAM (if used)',
     'VRAM freed via globals() cleanup above'),
    ('HF Inference API — check your usage',
     'https://huggingface.co/settings/billing'),
    ('No Vertex AI endpoints deployed in Part 4',
     'Part 4 uses prompting only, no deployed endpoints'),
]

for item, note in checks:
    print(f'  ✓ {item}')
    print(f'      → {note}')
    print()

print('FINAL STEP (manual):')
print('  → Colab menu: Runtime → Disconnect and delete runtime')
print('  → Verify no unexpected spend: https://console.cloud.google.com/billing')
print()
print('=' * 65)
print('ALL PARTS COMPLETE — MASTER SHUTDOWN CHECKLIST')
print('=' * 65)
master = [
    ('Part 1 runtime', 'CPU/small GPU — disconnect Colab runtime'),
    ('Part 2 checkpoints', 'Delete ckpt_part2_*/ if not done'),
    ('Part 3 checkpoints', 'Delete ckpt_part3_*/ if not done'),
    ('Part 4 Ollama', 'Stopped above'),
    ('Google Cloud billing', 'https://console.cloud.google.com/billing'),
    ('Colab runtime', 'Runtime → Disconnect and delete runtime'),
]
for resource, action in master:
    print(f'  □ {resource:30s}  {action}')
print()
print('Once all boxes are checked, you are fully shut down. No ongoing charges.')
